# 5.7 · 随机森林分类器 / Random Forest Classifier

> **课程定位 / Where this fits**
> 第 7 课，**Part 5 · 监督学习：分类**。
> Lesson 7, **Part 5 · Supervised Classification**.
>
> 5.6 结尾发现：单棵满树**高方差**（数据一扰动就长出完全不同的树）。随机森林的想法极其朴素——**训练很多棵彼此不太一样的树，让它们投票，把方差平均掉**。它几乎开箱即用、不怕过拟合、自带特征重要性和 OOB 评估，是 Kaggle 和工业界最常用的强基线之一。
> Section 5.6 ended on the fact that a single full tree is **high-variance**. Random Forest's idea is dead simple: **train many trees that differ from each other, let them vote, and average the variance away**. Near-zero tuning, hard to overfit, with built-in feature importance and OOB evaluation — one of the most-used strong baselines in Kaggle and industry.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $B$ —— 树的棵数 (`n_estimators`) / number of trees
> - bootstrap —— 有放回抽样得到的子样本 / a with-replacement subsample
> - $\rho$ —— 树之间预测的相关系数 / correlation between trees' predictions
> - $m$ (`max_features`) —— 每次分裂随机考虑的特征数 / features considered per split

> 💡 **面试相关 / Interview-relevant**
> - "bagging 为什么能降方差 / 数学原理"（★★★★★）
> - "随机森林的'随机'体现在哪两处"（★★★★★，样本 bootstrap + 特征子集）
> - "为什么要特征随机（只 bootstrap 不够）"（★★★★★，去相关）
> - "OOB 误差是什么"（★★★★）
> - "RF vs GBDT 区别"（★★★★，并行 bagging vs 串行 boosting）

---

## 学习目标 / Learning Objectives

1. 理解 bagging 降方差的数学（平均去相关的预测）。
   Understand the math of bagging variance reduction (averaging de-correlated predictions).
2. 理解随机森林的两处随机：**样本 bootstrap + 特征子集**。
   Understand RF's two sources of randomness.
3. 理解 **OOB** 免费验证。
   Understand the free **OOB** estimate.
4. 理解树的数量 / `max_features` 的影响。
   Understand the effect of tree count / `max_features`.
5. 对比 RF vs 单树 vs 预告 GBDT。
   Contrast RF vs single tree vs (preview) GBDT.

## 目录 / TOC
1. [先建直觉：三个臭皮匠](#1)
2. [bagging 降方差 ⭐](#2)
3. [两处随机：去相关 ⭐](#3)
4. [🚢 数据 + 单树 vs 森林](#4)
5. [OOB 免费验证 ⭐](#5)
6. [调参：n_estimators / max_features](#6)
7. [特征重要性 + RF vs GBDT](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 先建直觉：三个臭皮匠 / Intuition First

一棵决策树聪明但"情绪化"——换一批数据它就给出很不一样的答案（高方差）。但如果我们训练**很多棵略有差异的树**，让它们投票，**单棵树的随机抖动会互相抵消**，集体答案就稳得多。这就是"三个臭皮匠顶个诸葛亮"。
A single tree is smart but "moody" — feed it slightly different data and it gives quite different answers (high variance). But if we train **many slightly-different trees** and let them vote, **each tree's random wobble cancels out**, and the collective answer is far more stable. Wisdom of the crowd.

怎么让树"彼此不一样"？随机森林用**两种随机**：每棵树看**不同的样本**（bootstrap），每次分裂只看**随机的部分特征**。下面把"为什么平均能降方差"和"为什么要这两种随机"讲清楚。
How to make trees differ? Random Forest uses **two kinds of randomness**: each tree sees **different samples** (bootstrap), and each split considers only a **random subset of features**. Below we explain why averaging reduces variance and why both kinds of randomness are needed.


<a id="2"></a>
## 2. bagging 降方差 ⭐ / Why Bagging Reduces Variance

**Bagging** = Bootstrap AGGregating：对训练集**有放回抽样**得到 $B$ 个子集，每个训一棵树，预测时**投票/平均**。
**Bagging** = Bootstrap AGGregating: draw $B$ with-replacement subsets of the training data, train one tree on each, and **vote/average** at prediction.

**数学**：设 $B$ 个预测各自方差为 $\sigma^2$、两两相关系数为 $\rho$，它们平均后的方差是：
**Math:** if $B$ predictions each have variance $\sigma^2$ and pairwise correlation $\rho$, the variance of their average is:

$$\text{Var}\Big(\tfrac1B\sum_i\hat f_i\Big) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

- 当 $B\to\infty$，第二项 $\to 0$，只剩 $\rho\sigma^2$。
  As $B\to\infty$, the second term $\to 0$, leaving $\rho\sigma^2$.
- **关键**：光增加树的棵数 $B$ 还不够——方差下限被相关性 $\rho$ 卡住。**必须降低 $\rho$（让树彼此不同）**，这正是为什么需要特征随机（下一节）。
  **Key:** more trees alone isn't enough — the floor is set by $\rho$. We must **lower $\rho$ (make trees differ)**, which is why we add feature randomness (next section).

偏差几乎不变，方差大降——这就是 bagging 的核心。
Bias is basically unchanged, variance drops sharply — that's the heart of bagging.


<a id="3"></a>
## 3. 两处随机：去相关 ⭐ / Two Sources of Randomness

随机森林 = bagging 树 **+ 每次分裂只考虑随机的特征子集**：
Random Forest = bagging of trees **+ each split considers only a random subset of features**:

1. **样本随机 / sample randomness**：每棵树用一个 bootstrap 样本（大约含 63% 的不重复样本）。
   Each tree uses a bootstrap sample (≈63% unique samples).
2. **特征随机 / feature randomness**：每个分裂点只从随机的 $m$ 个特征里选最优（分类默认 $m=\sqrt{d}$）。
   Each split picks the best among $m$ random features (default $m=\sqrt{d}$ for classification).

为什么需要第 2 点：假如有一个超强特征（比如 Titanic 的 sex），那么只做 bootstrap 的话，每棵树都会先按它分裂 → 树长得高度相似（$\rho$ 大）→ 平均效果差。强制随机特征**逼不同的树去看不同的特征 → 去相关 → $\rho$ 降 → 方差降得更多**。
Why #2: if one feature is super-strong (e.g. Titanic's sex), bootstrap alone makes every tree split on it first → trees look alike (large $\rho$) → poor averaging. Forcing random features **makes different trees look at different features → de-correlation → lower $\rho$ → more variance reduction**.


<a id="4"></a>
## 4. 数据 + 单树 vs 森林 / Single Tree vs Forest

复用 **Titanic**（5.6 介绍并清洗过），同样的特征。先直观对比单棵满树和森林的测试表现与稳定性。
Reusing **Titanic** (introduced and cleaned in 5.6) with the same features. We first compare a single full tree vs a forest on test performance and stability.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

df = sns.load_dataset("titanic")
feat = ["pclass", "sex", "age", "sibsp", "parch", "fare"]
d = df[feat + ["survived"]].copy()
d["age"] = d["age"].fillna(d["age"].median())
d["fare"] = d["fare"].fillna(d["fare"].median())
d["sex"] = (d["sex"] == "male").astype(int)
from sklearn.model_selection import train_test_split, cross_val_score
X, y = d[feat].values, d["survived"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
tree = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)             # 单棵满树
rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_tr, y_tr)  # 300 棵树的森林
print(f"单棵满树 single full tree   test 准确率: {tree.score(X_te, y_te):.3f}")
print(f"随机森林 random forest       test 准确率: {rf.score(X_te, y_te):.3f}")
# 用交叉验证各折分数的标准差衡量"稳定性"(方差): 越小越稳
print(f"单树 5-fold CV 标准差 std: {cross_val_score(DecisionTreeClassifier(random_state=0),X,y,cv=5).std():.3f}")
print(f"森林 5-fold CV 标准差 std: {cross_val_score(RandomForestClassifier(100,random_state=0),X,y,cv=5).std():.3f} (更稳 more stable)")


<a id="5"></a>
## 5. OOB 免费验证 ⭐ / Out-of-Bag Estimate

每棵树的 bootstrap 样本只用到了约 63% 的数据，**剩下约 37% 是这棵树"没见过"的（袋外, out-of-bag）**。于是对每个样本，我们可以只用"没见过它的那些树"来预测它，汇总起来就得到一个**几乎免费的验证误差**——不用单独划一块验证集。
Each tree's bootstrap uses ≈63% of the data, so **≈37% is "out-of-bag" (unseen) for that tree**. For each sample we can predict it using only the trees that never saw it, giving an **almost-free validation error** — no separate hold-out set needed.


In [ ]:
# oob_score=True 让森林在训练时顺便用袋外样本算验证准确率
rf_oob = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0).fit(X_tr, y_tr)
print(f"OOB 准确率 OOB accuracy: {rf_oob.oob_score_:.3f}")
print(f"测试准确率 test accuracy: {rf_oob.score(X_te, y_te):.3f}")
print("两者接近 → OOB 是可靠的免费验证, 小数据尤其有用(省下留出集) / OOB ≈ test error, free")

# 验证 ~37% 袋外规则: 有放回抽 n 次, 大约只覆盖 63% 的样本 / verify the ~37% rule
n = 10000; rng = np.random.default_rng(0)
sampled = np.unique(rng.integers(0, n, n))    # 抽 n 次后去重, 看覆盖了多少不同样本
print(f"\nbootstrap 覆盖率 coverage: {len(sampled)/n:.1%} (理论 1-1/e≈63.2%), 袋外 OOB≈36.8%")


<a id="6"></a>
## 6. 调参：n_estimators / max_features / Tuning

随机森林几乎不用调参，两个主要旋钮：
RF needs almost no tuning; two main knobs:
- **n_estimators（树的棵数）**：越多越稳，**不会过拟合**（只会收敛后变慢）——这点和 GBDT 相反。
  More trees = more stable and **never overfits** (just converges, then gets slower) — opposite of GBDT.
- **max_features（每次分裂的随机特征数）**：越小 → 树越去相关（方差更低），但太小 → 单棵树太弱（偏差升高）。是去相关与单树强度之间的权衡。
  Smaller = more de-correlated trees (lower variance), but too small = each tree too weak (higher bias). A trade-off.


In [ ]:
# 树越多越稳, 收敛后不再过拟合 / more trees converge and never overfit
ns = [1, 5, 10, 25, 50, 100, 200, 400]
accs = [RandomForestClassifier(n, random_state=0).fit(X_tr,y_tr).score(X_te,y_te) for n in ns]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(ns, accs, "o-"); axes[0].set_xlabel("n_estimators 树的棵数"); axes[0].set_ylabel("test 准确率")
axes[0].set_title("树越多 → 收敛后平稳(不会过拟合) / more trees → converge, no overfit")

# max_features 越小, 树之间越去相关; 太小则单树太弱 / max_features controls de-correlation
mfs = [1, 2, 3, 4, 5, 6]
mf_acc = [cross_val_score(RandomForestClassifier(200, max_features=m, random_state=0),X,y,cv=5).mean()
          for m in mfs]
axes[1].plot(mfs, mf_acc, "s-"); axes[1].axvline(np.sqrt(6), color="r", ls="--", label="√d 默认 default")
axes[1].set_xlabel("max_features 每次分裂的特征数"); axes[1].set_ylabel("CV 准确率"); axes[1].legend()
axes[1].set_title("max_features 小→树更去相关; 太小→单树太弱")
plt.tight_layout(); plt.show()
print("n_estimators 越多越好(只是变慢); max_features 控制去相关 vs 单树强度的权衡")


<a id="7"></a>
## 7. 特征重要性 + RF vs GBDT / Importance & vs GBDT

RF 的**特征重要性** = 所有树上该特征带来的平均不纯度下降。**但要注意它有偏**：不纯度重要性偏向高基数/连续特征（它们能切出更多分裂点）。更可靠的是**置换重要性(permutation importance)**——把某一列的值打乱，看模型性能掉多少（掉得越多越重要）。
RF **feature importance** = average impurity decrease per feature across trees. **But beware its bias**: impurity importance favors high-cardinality/continuous features (they offer more split points). More reliable is **permutation importance** — shuffle one column and see how much performance drops (bigger drop = more important).


In [ ]:
from sklearn.inspection import permutation_importance
imp_gini = pd.Series(rf.feature_importances_, index=feat)         # 不纯度重要性(训练时自带)
# 置换重要性: 在测试集上把每个特征打乱 n_repeats 次, 测准确率平均下降多少
perm = permutation_importance(rf, X_te, y_te, n_repeats=20, random_state=0)
imp_perm = pd.Series(perm.importances_mean, index=feat)
cmp = pd.DataFrame({"不纯度重要性 impurity": imp_gini,
                    "置换重要性 permutation": imp_perm}).sort_values("置换重要性 permutation", ascending=False)
print(cmp.round(3).to_string())
print("\nsex/pclass 最关键; 两种重要性大体一致但不完全相同")
print("(不纯度重要性对连续特征 fare/age 略偏高 → 用置换重要性更可靠)")


**RF vs GBDT(5.8)** —— 两大集成范式对比（面试高频）：
**RF vs GBDT (5.8)** — the two big ensemble paradigms (frequently asked):

| | 随机森林 Random Forest (Bagging) | GBDT (Boosting) |
|---|---|---|
| 训练 training | **并行**，树相互独立 / parallel, independent | **串行**，每棵纠正前面的残差 / sequential, fixes prior errors |
| 目标 goal | 降**方差**（用深树）/ reduce variance (deep trees) | 降**偏差**（用浅树）/ reduce bias (shallow trees) |
| 过拟合 overfit | 树多不会过拟合 / more trees ≠ overfit | 树多**会**过拟合 / more trees can overfit |
| 调参 tuning | 少，开箱即用 / minimal | 多，需细调 / more, careful |
| 通常精度 accuracy | 强 strong | 调好后**更强**（表格之王）/ stronger when tuned |


<a id="8"></a>
## 8. 小结 / Summary

```
随机森林 = bagging 树 + 特征随机; 投票降方差
bagging 数学: Var = ρσ² + (1-ρ)/B σ²; 加树消第二项, 但被 ρ 卡住 → 必须去相关
两处随机: 样本 bootstrap(~63%) + 每分裂随机 √d 个特征 → 降 ρ
OOB: ~37% 袋外样本做免费验证, ≈测试误差
树越多越稳(不过拟合); max_features 控去相关; 不纯度重要性偏连续特征→用置换重要性
RF(并行/降方差) vs GBDT(串行/降偏差)
```

### 💡 面试速查 / Interview cheat-sheet
1. **bagging 降方差**：平均去相关预测；公式 $\rho\sigma^2+(1-\rho)\sigma^2/B$。
   Bagging reduces variance by averaging de-correlated predictions.
2. **两处随机**：bootstrap 样本 + 随机特征子集；特征随机是为**去相关**（光 bootstrap 不够）。
   Two randomnesses: bootstrap + random features; feature randomness is for de-correlation.
3. **OOB ≈ 免费交叉验证**（约 37% 袋外）。
   OOB ≈ free cross-validation (~37% out-of-bag).
4. **树多不过拟合**（与 GBDT 相反）。
   More trees don't overfit (opposite of GBDT).
5. **不纯度重要性有偏** → 置换重要性更可靠。
   Impurity importance is biased → prefer permutation importance.

### 下一节 / Next
**5.8 GBDT**——从 bagging 切到 boosting。不再独立平均，而是**一棵接一棵地纠正前面的错误**，表格数据之王的起点。
**5.8 GBDT** — from bagging to boosting. Trees no longer average independently but **correct each other's errors in sequence**; the start of the table-data champions.
